# Tool Calling Fine-Tuning: QLoRA (NF4 4-bit) on T4 GPU
Bu notebook, **Qwen2.5-0.5B** modeli uzerinde **QLoRA (NF4 4-bit Kuantizasyon + LoRA)** yontemiyle Tool / Function Calling yetenegini egitmek ve degerlendirmek icin hazirlanmistir.

- **Metot:** QLoRA (NF4 4-bit, r=16, alpha=32, target_modules=[q_proj, k_proj, v_proj, o_proj])
- **Taban Model:** Qwen/Qwen2.5-0.5B (NF4 4-bit, compute_dtype=float16)
- **Hedef Donanim:** Google Colab T4 (16GB VRAM, float16)
- **Sekans Uzunlugu:** max_seq_len = 2048
- **Bellek Optimizasyonu:** `batch_size: 2`, `grad_accum_steps: 4` (efektif batch = 8), Gradient Checkpointing aktif.

---

### 1. GPU ve Donanim Kontrolu

In [ ]:
!nvidia-smi

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"bf16 support: {torch.cuda.is_bf16_supported()}")

### 2. Projeyi Klonla ve Guncelle

In [ ]:
import os

REPO_URL = "https://github.com/fatihkadim/tool-calling-ft.git"
PROJECT_DIR = "tool-calling-ft"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}

%cd {PROJECT_DIR}
!git pull origin main
!pwd

### 3. Bagimliliklarin Kurulumu

In [ ]:
!pip install -q --upgrade pip
!pip install -q "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "datasets>=3.0" "trl>=0.11" "accelerate>=1.0" pyyaml tqdm pandas matplotlib

# uv_build backend'ini kur, sonra projeyi editable olarak yukle
!pip install -q "uv_build>=0.11.7,<0.12.0"
!pip install -q --no-build-isolation -e .

### 4. Veri Setini Hazirla (Gerekirse)

In [ ]:
!python -m tool_calling_ft.data.prepare_dataset

### 5. T4 Icin QLoRA Config Olustur

In [ ]:
import yaml

config = {
    "method": "qlora",
    "base_model": "Qwen/Qwen2.5-0.5B",
    "dataset": "NousResearch/hermes-function-calling-v1",
    "output_dir": "checkpoints/qlora",
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",  # T4 bf16 desteklemiyor
        "bnb_4bit_use_double_quant": True,
    },
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "training": {
        "epochs": 3,
        "batch_size": 2,
        "grad_accum_steps": 4,
        "learning_rate": 2e-4,
        "max_seq_len": 2048,
        "warmup_ratio": 0.05,
        "save_steps": 200,
        "seed": 42,
    },
}

with open("configs/qlora_t4.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("T4 QLoRA config yazildi: configs/qlora_t4.yaml")
print(yaml.dump(config, default_flow_style=False))

### 6. QLoRA Egitimini Baslat

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.training.train --config configs/qlora_t4.yaml

### 7. Canli Demo (Inference Testi)

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = "checkpoints/qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

system_prompt = """You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags.
<tools>
[{"type": "function", "function": {"name": "get_current_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}}]
</tools>
For each function call return a json object with function name and arguments within <tool_call> </tool_call> tags."""

user_query = "What is the weather in Tokyo in celsius?"
prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_ids = [tokenizer.eos_token_id]
if isinstance(im_end_id, int) and im_end_id != tokenizer.eos_token_id:
    stop_ids.append(im_end_id)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.1,
        eos_token_id=stop_ids,
    )

response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("=" * 50)
print("QLoRA MODEL CIKTISI:")
print(response.strip())
print("=" * 50)

### 8. QLoRA Degerlendirme (Evaluation)

In [ ]:
# QLoRA Modelini Degerlendir:
!CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.eval.harness \
    --method qlora \
    --adapter checkpoints/qlora \
    --dataset data/processed/eval_subset.jsonl

### 9. Sonuclari Yedekle (Google Drive)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/tool_calling_qlora_results
!cp -r checkpoints/qlora /content/drive/MyDrive/tool_calling_qlora_results/ 2>/dev/null || echo 'checkpoints klasoru yok'
!cp -r reports /content/drive/MyDrive/tool_calling_qlora_results/ 2>/dev/null || echo 'reports klasoru yok'
print("QLoRA Checkpoint ve raporlar Drive'a kaydedildi!")